# 02 — Cube + WCS + catalog alignment

**§11 step-2 gate.** For each downloaded THINGS cube:
* render the moment-0 (integrated intensity) map
* overlay every B11 hole as a circle of radius `d/2` (in arcsec)
* eyeball: the catalog circles must land in the visible cavities.

Refuse to proceed past Gate 2 if any galaxy fails this check — it
means our coordinate transforms are off and every downstream window
extraction will be wrong.

In [ ]:
%matplotlib inline
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle

from hishells.catalog import load_catalog
from hishells.cubes import (
    Cube,
    cube_stats,
    load_cube,
    moment0,
    world_to_pix,
)

In [ ]:
REPO = Path('..').resolve()
DATA = REPO / 'Data' / 'THINGS'
CACHE = REPO / 'results' / 'cube_stats.json'

cat = load_catalog(REPO / 'Data' / 'J_AJ_141_23')
available = sorted(p.name.replace('_NA_CUBE_THINGS.FITS', '')
                   for p in DATA.glob('*_NA_CUBE_THINGS.FITS'))
print(f'{len(available)} cubes downloaded:', available)

In [ ]:
def overlay_check(galaxy_id: str, cube: Cube, ax) -> int:
    """Render moment-0 with B11 holes overlaid; return holes plotted."""
    m0 = moment0(cube)
    holes = cat.holes[cat.holes['galaxy_id'] == galaxy_id]
    xs, ys = world_to_pix(cube, holes['ra_deg'].values, holes['dec_deg'].values)
    radii_pix = (holes['diameter_arcsec'].values / 2.0) / cube.pixel_scale_arcsec

    vmin, vmax = np.nanpercentile(m0, [2, 99.5])
    ax.imshow(m0, origin='lower', cmap='magma', vmin=vmin, vmax=vmax, interpolation='nearest')
    for x, y, r, ht in zip(xs, ys, radii_pix, holes['hole_type']):
        color = {1: 'cyan', 2: 'lime', 3: 'red'}[int(ht)]
        ax.add_patch(Circle((x, y), r, fill=False, edgecolor=color, lw=0.6, alpha=0.85))
    ax.set_title(f'{galaxy_id}  ({len(holes)} holes)  beam={cube.beam_bmaj_arcsec:.1f}"')
    ax.set_xticks([]); ax.set_yticks([])
    return len(holes)

In [ ]:
stats_rows = []
n_per_row = 3
rows = (len(available) + n_per_row - 1) // n_per_row
fig, axes = plt.subplots(rows, n_per_row, figsize=(5 * n_per_row, 5 * rows))
axes = np.atleast_1d(axes).ravel()
for ax, gid in zip(axes, available):
    cube = load_cube(DATA / f'{gid}_NA_CUBE_THINGS.FITS')
    overlay_check(gid, cube, ax)
    stats_rows.append(cube_stats(cube, cache_path=CACHE))
for ax in axes[len(available):]:
    ax.set_visible(False)
plt.tight_layout()

## Per-cube stats table

Cached to `results/cube_stats.json` so later notebooks (and
`scripts/train_logo.py`) can avoid re-loading every cube just to read
its noise level.

In [ ]:
import pandas as pd
df = pd.DataFrame(stats_rows).set_index('galaxy_id').sort_index()
df

## Gate decision

Manually inspect each subplot above. If any galaxy's circles do not
land in cavities of the moment-0 map, stop and debug the WCS pipeline
in `hishells/cubes.py:world_to_pix` and the catalog parsing in
`hishells/catalog.py:_parse_table7_line` before continuing to Gate 3.